# Notebook 1: CWRU Data Pipeline + Feature Extraction
**IEEE IES GenAI Challenge 2026 — Safety-Constrained TTC for Industrial Anomaly Attribution**

This notebook:
1. Downloads and parses the CWRU bearing dataset
2. Segments signals into windows
3. Extracts time-domain + frequency-domain features
4. Formats features as natural-language strings for LLM input
5. Saves the feature dataset as `cwru_features.pkl` for Notebooks 2 & 3

**Fault classes:** Normal (0), Inner Race (1), Outer Race (2), Ball (3)

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install scipy numpy pandas scikit-learn requests -q

In [ ]:
import os
import pickle
import urllib.request
import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy.fft import fft, fftfreq
from scipy.stats import kurtosis, skew
from sklearn.model_selection import train_test_split
from pathlib import Path

print('Dependencies loaded.')

## 1. Download CWRU .mat Files

We use drive-end (DE) vibration signals at 12 kHz, 1797 RPM (0 HP load) for four fault conditions at 0.007" fault diameter (the lightest severity — most realistic for early-fault attribution).

In [ ]:
# CWRU files: (filename, fault_label, fault_class_id)
# Source: Case Western Reserve University Bearing Data Center
CWRU_FILES = [
    # Normal baseline
    ('97.mat',  'Normal',      0),
    # Inner race fault — 0.007" diameter
    ('105.mat', 'Inner Race',  1),
    # Ball fault — 0.007" diameter
    ('118.mat', 'Ball',        3),
    # Outer race fault — 0.007" diameter, centered @6
    ('130.mat', 'Outer Race',  2),
]

BASE_URL = 'https://engineering.case.edu/sites/default/files/'
DATA_DIR = Path('cwru_data')
DATA_DIR.mkdir(exist_ok=True)

def download_file(filename):
    dest = DATA_DIR / filename
    if dest.exists():
        print(f'  {filename} already exists, skipping.')
        return
    url = BASE_URL + filename
    print(f'  Downloading {filename}...')
    try:
        urllib.request.urlretrieve(url, dest)
        print(f'  Done.')
    except Exception as e:
        print(f'  FAILED: {e}')
        print(f'  → Manually download from: https://engineering.case.edu/bearingdatacenter/download-data-file')
        print(f'    Then upload to Colab at: cwru_data/{filename}')

print('Downloading CWRU files...')
for fname, label, cid in CWRU_FILES:
    download_file(fname)
print('Done.')

### Manual upload fallback
If the download fails (CWRU sometimes blocks automated requests), run this cell to upload files manually from your machine.

In [ ]:
# OPTIONAL: Only run this if automatic download failed above
# from google.colab import files
# uploaded = files.upload()  # upload 97.mat, 105.mat, 118.mat, 130.mat
# import shutil
# for fname in uploaded:
#     shutil.move(fname, DATA_DIR / fname)
# print('Files moved to cwru_data/')

## 2. Parse .mat Files → Raw Signal Segments

In [ ]:
WINDOW_SIZE = 1024   # samples per segment (~85ms at 12kHz)
OVERLAP     = 0.5    # 50% overlap between windows
STRIDE      = int(WINDOW_SIZE * (1 - OVERLAP))
FS          = 12000  # sampling frequency (Hz)

def extract_de_signal(mat_path):
    """Extract drive-end accelerometer signal from CWRU .mat file."""
    mat = loadmat(mat_path)
    # CWRU key format: X{id}_DE_time
    de_keys = [k for k in mat.keys() if 'DE_time' in k]
    if not de_keys:
        raise ValueError(f'No DE_time key found in {mat_path}. Keys: {list(mat.keys())}')
    signal = mat[de_keys[0]].flatten()
    return signal

def segment_signal(signal, window_size, stride):
    """Slice signal into overlapping windows."""
    segments = []
    start = 0
    while start + window_size <= len(signal):
        segments.append(signal[start:start + window_size])
        start += stride
    return np.array(segments)

all_segments = []
all_labels   = []
all_label_names = []

for fname, label_name, label_id in CWRU_FILES:
    mat_path = DATA_DIR / fname
    if not mat_path.exists():
        print(f'MISSING: {fname} — skipping. Upload manually.')
        continue
    signal = extract_de_signal(mat_path)
    segments = segment_signal(signal, WINDOW_SIZE, STRIDE)
    all_segments.append(segments)
    all_labels.extend([label_id] * len(segments))
    all_label_names.extend([label_name] * len(segments))
    print(f'{label_name:12s} | {fname} | signal length: {len(signal):,} | segments: {len(segments)}')

all_segments = np.vstack(all_segments)
all_labels   = np.array(all_labels)
print(f'\nTotal segments: {len(all_segments)} | Shape: {all_segments.shape}')

## 3. Feature Extraction

We extract 14 features per segment across two domains:
- **Time domain (8):** RMS, peak, crest factor, kurtosis, skewness, variance, shape factor, impulse factor
- **Frequency domain (6):** dominant frequency, dominant magnitude, spectral centroid, spectral spread, FFT energy in 3 diagnostic bands (BPFI, BPFO, BSF regions)

These are the same statistical features used by Tao et al. (2024) and are compatible with LLM textualization.

In [ ]:
# Approximate fault frequency bands for CWRU DE bearings at 1797 RPM
# BPFI ≈ 162 Hz, BPFO ≈ 107 Hz, BSF ≈ 141 Hz (±15 Hz bands)
FAULT_BANDS = {
    'BPFI_band': (147, 177),
    'BPFO_band': (92,  122),
    'BSF_band':  (126, 156),
}

def extract_features(segment, fs=FS):
    """Extract 14 time + frequency domain features from a signal segment."""
    # ── Time domain ──────────────────────────────────────────────────────────
    rms          = np.sqrt(np.mean(segment**2))
    peak         = np.max(np.abs(segment))
    crest_factor = peak / (rms + 1e-9)
    kurt         = float(kurtosis(segment))
    skewness     = float(skew(segment))
    variance     = float(np.var(segment))
    mean_abs     = float(np.mean(np.abs(segment)))
    shape_factor = rms / (mean_abs + 1e-9)
    impulse_factor = peak / (mean_abs + 1e-9)

    # ── Frequency domain ─────────────────────────────────────────────────────
    N       = len(segment)
    freqs   = fftfreq(N, d=1/fs)[:N//2]
    magnitudes = np.abs(fft(segment))[:N//2] / N

    dom_idx        = np.argmax(magnitudes)
    dom_freq       = float(freqs[dom_idx])
    dom_magnitude  = float(magnitudes[dom_idx])

    # Spectral centroid and spread
    total_energy   = np.sum(magnitudes) + 1e-9
    spec_centroid  = float(np.sum(freqs * magnitudes) / total_energy)
    spec_spread    = float(np.sqrt(np.sum(((freqs - spec_centroid)**2) * magnitudes) / total_energy))

    # Energy in fault frequency bands
    band_energies = {}
    for band_name, (f_low, f_high) in FAULT_BANDS.items():
        mask = (freqs >= f_low) & (freqs <= f_high)
        band_energies[band_name] = float(np.sum(magnitudes[mask]**2))

    return {
        'rms':            rms,
        'peak':           peak,
        'crest_factor':   crest_factor,
        'kurtosis':       kurt,
        'skewness':       skewness,
        'variance':       variance,
        'shape_factor':   shape_factor,
        'impulse_factor': impulse_factor,
        'dom_freq_hz':    dom_freq,
        'dom_magnitude':  dom_magnitude,
        'spec_centroid':  spec_centroid,
        'spec_spread':    spec_spread,
        **band_energies
    }

print('Extracting features from all segments...')
features_list = []
for i, seg in enumerate(all_segments):
    features_list.append(extract_features(seg))
    if (i+1) % 200 == 0:
        print(f'  Processed {i+1}/{len(all_segments)}')

df = pd.DataFrame(features_list)
df['label']      = all_labels
df['label_name'] = all_label_names
print(f'\nFeature DataFrame shape: {df.shape}')
df.head()

## 4. Dataset Split

We sample 200 test samples (50 per class, stratified) for M2 preliminary results. The remaining data is split into train (for few-shot example selection) and an unlabeled pool (for Experiment 4 in M3).

In [ ]:
# Stratified sample: 50 per class for test set
TEST_PER_CLASS = 50
FEWSHOT_PER_CLASS = 3   # used in Notebook 2 for prompting

test_indices   = []
train_indices  = []

for label_id in [0, 1, 2, 3]:
    class_idx = df.index[df['label'] == label_id].tolist()
    np.random.seed(42)
    np.random.shuffle(class_idx)
    test_indices.extend(class_idx[:TEST_PER_CLASS])
    train_indices.extend(class_idx[TEST_PER_CLASS:])

df_test  = df.loc[test_indices].reset_index(drop=True)
df_train = df.loc[train_indices].reset_index(drop=True)

print(f'Test set:  {len(df_test)} samples')
print(df_test['label_name'].value_counts())
print(f'\nTrain set: {len(df_train)} samples')
print(df_train['label_name'].value_counts())

## 5. Feature Textualization

Convert each feature vector into a natural-language description string. This bridges the modality gap between numerical sensor data and LLM input (Layer 1 of the framework).

In [ ]:
def textualize_features(row):
    """Convert a feature row into a natural-language vibration signal description."""
    text = (
        f"Bearing vibration signal analysis (CWRU drive-end, 12kHz, 1797 RPM):\n"
        f"- RMS amplitude: {row['rms']:.4f} g\n"
        f"- Peak amplitude: {row['peak']:.4f} g\n"
        f"- Crest factor: {row['crest_factor']:.2f} (ratio of peak to RMS; >3 suggests impulsive events)\n"
        f"- Kurtosis: {row['kurtosis']:.3f} (>3 indicates impulsive/non-Gaussian behavior)\n"
        f"- Skewness: {row['skewness']:.3f}\n"
        f"- Shape factor: {row['shape_factor']:.3f}\n"
        f"- Impulse factor: {row['impulse_factor']:.3f}\n"
        f"- Dominant frequency: {row['dom_freq_hz']:.1f} Hz (magnitude: {row['dom_magnitude']:.5f})\n"
        f"- Spectral centroid: {row['spec_centroid']:.1f} Hz | Spectral spread: {row['spec_spread']:.1f} Hz\n"
        f"- Energy in BPFI band (147-177 Hz, inner race): {row['BPFI_band']:.6f}\n"
        f"- Energy in BPFO band (92-122 Hz, outer race): {row['BPFO_band']:.6f}\n"
        f"- Energy in BSF band (126-156 Hz, ball fault): {row['BSF_band']:.6f}"
    )
    return text

df_test['text']  = df_test.apply(textualize_features, axis=1)
df_train['text'] = df_train.apply(textualize_features, axis=1)

# Preview one example
print('=== Example Textualized Feature String ===')
print(df_test.iloc[0]['text'])
print(f'\nLabel: {df_test.iloc[0]["label_name"]}')

## 6. Build Few-Shot Examples Pool

Select 3 examples per fault class from the training set. These will be injected into every LLM prompt in Notebooks 2 & 3.

In [ ]:
LABEL_ID_TO_NAME = {0: 'Normal', 1: 'Inner Race', 2: 'Outer Race', 3: 'Ball'}

fewshot_examples = []
for label_id, label_name in LABEL_ID_TO_NAME.items():
    pool = df_train[df_train['label'] == label_id]
    sampled = pool.sample(n=FEWSHOT_PER_CLASS, random_state=42)
    for _, row in sampled.iterrows():
        fewshot_examples.append({
            'text':       row['text'],
            'label':      label_id,
            'label_name': label_name
        })

print(f'Few-shot pool: {len(fewshot_examples)} examples ({FEWSHOT_PER_CLASS} per class)')
print('Class distribution:', {e['label_name'] for e in fewshot_examples})

## 7. Save Artifacts

In [ ]:
SAVE_PATH = Path('cwru_features.pkl')

artifact = {
    'df_test':          df_test,
    'df_train':         df_train,
    'fewshot_examples': fewshot_examples,
    'label_map':        LABEL_ID_TO_NAME,
    'config': {
        'window_size':        WINDOW_SIZE,
        'overlap':            OVERLAP,
        'fs':                 FS,
        'test_per_class':     TEST_PER_CLASS,
        'fewshot_per_class':  FEWSHOT_PER_CLASS,
        'fault_bands':        FAULT_BANDS,
    }
}

with open(SAVE_PATH, 'wb') as f:
    pickle.dump(artifact, f)

print(f'Saved to {SAVE_PATH}')
print(f'File size: {SAVE_PATH.stat().st_size / 1024:.1f} KB')
print('\n=== Summary ===')
print(f'Test samples:  {len(df_test)} ({TEST_PER_CLASS} per class x 4 classes)')
print(f'Train samples: {len(df_train)}')
print(f'Features:      14 (8 time-domain, 6 frequency-domain)')
print(f'Few-shot pool: {len(fewshot_examples)} examples')
print('\nReady for Notebook 2 (static baseline) and Notebook 3 (TTC loop).')

In [ ]:
# Optional: Download the artifact from Colab
# from google.colab import files
# files.download('cwru_features.pkl')